In [14]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

client = Client("IRIS")

inv = client.get_stations(
    latitude=46.8529,
    longitude=-121.7604,
    maxradius=0.45,        # degrees — adjust to your target radius
    starttime=UTCDateTime("2010-01-01"),
    endtime=UTCDateTime("2026-01-01"),
    level="station"
)
for net in inv:
    for sta in net:
        print(f"{net.code}.{sta.code:6s}  "
              f"{sta.start_date} → {sta.end_date}")

C0.PALI    2025-09-08T00:00:00.000000Z → None
CC.ARAT    2020-08-06T00:00:00.000000Z → None
CC.CARB    2018-10-16T00:00:00.000000Z → None
CC.COPP    2022-09-20T11:00:00.000000Z → None
CC.CRBN    2020-10-22T00:00:00.000000Z → None
CC.CRYS    2025-09-11T00:00:00.000000Z → None
CC.ELBE    2023-08-23T00:00:00.000000Z → None
CC.GNOB    2022-10-18T13:21:00.000000Z → None
CC.GOBB    2022-09-07T00:00:00.000000Z → None
CC.GRWR    2025-09-08T00:00:00.000000Z → None
CC.GTWY    2020-10-28T20:00:00.000000Z → None
CC.KAUT    2020-09-02T00:00:00.000000Z → None
CC.KAVK    2023-08-22T00:00:00.000000Z → None
CC.LONE    2025-09-08T00:00:00.000000Z → None
CC.LONR    2025-10-02T16:30:00.000000Z → None
CC.MILD    2022-09-20T00:00:00.000000Z → None
CC.MIRR    2020-07-14T00:00:00.000000Z → 2022-09-04T00:00:00.000000Z
CC.OBSR    2008-09-10T00:00:00.000000Z → None
CC.OPCH    2020-10-26T00:00:00.000000Z → None
CC.PALI    2025-09-08T00:00:00.000000Z → None
CC.PANH    2007-09-10T00:00:00.000000Z → None
CC.PARA    

In [19]:
# Quick completeness check — run this first
import pandas as pd
import os
from datetime import date, timedelta

catalog_dir = "../logs/mt_rainier_common_detections"
start = date(2010, 1, 1)
end = date(2026, 1, 1)

missing = []
current = start
while current <= end:
    fname = f"common_{current.strftime('%Y%m%d')}_0000_to_{current.strftime('%Y%m%d')}_2359_events.csv"
    if not os.path.exists(os.path.join(catalog_dir, fname)):
        missing.append(current)
    current += timedelta(days=1)

print(f"Missing days: {len(missing)}")
print(f"First 10: {missing[:10]}")

Missing days: 0
First 10: []


In [23]:
import pandas as pd
import glob

files = glob.glob("../logs/mt_rainier_common_detections/common_*.csv")
dfs = []
for f in files:
    df = pd.read_csv(f)
    df["date"] = pd.to_datetime(df["rounded_start"], format='ISO8601').dt.date
    dfs.append(df)

catalog = pd.concat(dfs, ignore_index=True)
catalog["rounded_start"] = pd.to_datetime(catalog["rounded_start"], format='ISO8601')
catalog["year"] = catalog["rounded_start"].dt.year
catalog["month"] = catalog["rounded_start"].dt.month
catalog["doy"] = catalog["rounded_start"].dt.dayofyear

# Class breakdown
print(catalog["most_common_class"].value_counts())

# Annual counts
print(catalog.groupby(["year","most_common_class"]).size().unstack())

/tmp/ipykernel_665886/2526448294.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  catalog = pd.concat(dfs, ignore_index=True)


most_common_class
su    399377
eq    202234
px     55362
Name: count, dtype: int64
most_common_class     eq    px     su
year                                 
2010                2065  1298   4382
2011                6314  3284  15611
2012                3838  2998  12973
2013               10811  3801  21001
2014                4818  2284  11181
2015                3088  2218  14804
2016                1989  1224   7292
2017                3557  1930  11502
2018                6119  2221  14691
2019               14737  3582  27372
2020               11552  4043  28911
2021               12262  3313  25873
2022                9831  3037  27841
2023               23866  5441  50477
2024               32944  6560  63212
2025               51839  7328  54645
2026                2604   800   7609


In [29]:
pd.set_option("display.max_columns", None)
comp_cat = pd.read_csv('../src/catalog_output/master_catalog.csv')
comp_cat.head()

,cluster_id,rounded_start,num_stations,stations,all_classes,most_common_class,mean_auc,mean_max,mean_prob,file_date,date,year,month,hour,doy,frac_eq,frac_su,frac_px,frac_noise,unanimous,vote_margin,auc_is_summed,mean_auc_per_station,low_confidence,high_confidence,ambiguous_class,event_id
0,1,2010-01-01 00:00:50.000400+00:00,4,"['FMW', 'RCM', 'RER', 'RVC']","['px', 'su', 'px', 'su']",px,32.506553,0.875505,0.511687,2010-01-01,2010-01-01,2010,1,0,1,0.000000,0.500000,0.500000,0.0,False,0.500000,True,8.126638,False,False,True,MR0000000
1,7,2010-01-01 00:16:50.012000+00:00,4,"['FMW', 'LON', 'RCS', 'RER']","['eq', 'px', 'eq', 'eq']",eq,4.498854,0.722635,0.416221,2010-01-01,2010-01-01,2010,1,0,1,0.750000,0.000000,0.250000,0.0,False,0.750000,True,1.124714,False,False,False,MR0000001
2,41,2010-01-01 01:27:40.000400+00:00,3,"['RCS', 'RER', 'RVC']","['px', 'eq', 'eq']",eq,4.266211,0.695121,0.405472,2010-01-01,2010-01-01,2010,1,1,1,0.666667,0.000000,0.333333,0.0,False,0.666667,True,1.422070,False,False,False,MR0000002
3,97,2010-01-01 03:41:30.000400+00:00,3,"['RCM', 'RCS', 'RVC']","['su', 'px', 'eq']",eq,26.212075,0.638117,0.385640,2010-01-01,2010-01-01,2010,1,3,1,0.333333,0.333333,0.333333,0.0,False,0.333333,True,8.737358,True,False,True,MR0000003
4,102,2010-01-01 03:56:30.000400+00:00,5,"['FMW', 'LO2', 'LON', 'RCS', 'RER']","['eq', 'su', 'eq', 'eq', 'eq']",eq,13.542234,0.751031,0.403142,2010-01-01,2010-01-01,2010,1,3,1,0.800000,0.200000,0.000000,0.0,False,0.800000,True,2.708447,False,True,False,MR0000004


In [30]:
len(comp_cat)

656973

In [24]:
# Check how many stations are active per year
import pandas as pd
import glob
import ast

files = glob.glob("../logs/mt_rainier_common_detections/common_*.csv")

rows = []
for f in files:
    df = pd.read_csv(f)
    if df.empty:
        continue
    df["rounded_start"] = pd.to_datetime(df["rounded_start"], format='ISO8601')
    year = df["rounded_start"].dt.year.iloc[0]
    # flatten all stations seen this day
    all_stations = set()
    for s in df["stations"]:
        try:
            all_stations.update(ast.literal_eval(s))
        except:
            pass
    rows.append({"year": year, "n_stations": len(all_stations)})

station_df = pd.DataFrame(rows).groupby("year")["n_stations"].max()
print(station_df)

year
2010    10
2011    13
2012    10
2013    10
2014    10
2015    10
2016     9
2017    14
2018    31
2019    18
2020    24
2021    24
2022    31
2023    32
2024    41
2025    35
2026    34
Name: n_stations, dtype: int64


In [17]:
from obspy.clients.fdsn import Client
import json

STATIONS = [
    ("CC","ARAT"),("CC","CARB"),("CC","COPP"),("CC","CRBN"),("CC","CRYS"),
    ("CC","GNOB"),("CC","GOBB"),("CC","GTWY"),("CC","KAUT"),("CC","KAVK"),
    ("CC","LONE"),("CC","LONR"),("CC","MILD"),("CC","OBSR"),("CC","OPCH"),
    ("CC","PANH"),("CC","PARA"),("CC","PR01"),("CC","PR02"),("CC","PR03"),
    ("CC","PR04"),("CC","PR05"),("CC","RUSH"),("CC","SIFT"),("CC","TABR"),
    ("CC","TAVI"),("CC","VOIT"),("CC","WOW"),
    ("UW","FMW"),("UW","LO2"),("UW","LON"),("UW","RCM"),
    ("UW","RCS"),("UW","RER"),("UW","STAR"),
]

client = Client("IRIS")
results = []
for net, sta in STATIONS:
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        for n in inv:
            for s in n:
                results.append({"name": f"{net}.{sta}",
                                "lat": s.latitude, "lon": s.longitude})
                print(f"{net}.{sta}  {s.latitude:.4f}  {s.longitude:.4f}")
                break
    except Exception as e:
        print(f"FAIL {net}.{sta}: {e}")

print("\nJSON:")
print(json.dumps(results))

CC.ARAT  46.7890  -121.8530
CC.CARB  46.9883  -122.0054
CC.COPP  46.7980  -121.8287
CC.CRBN  46.9882  -121.9610
CC.CRYS  46.9344  -121.5003
CC.GNOB  46.7941  -121.9144
CC.GOBB  46.7941  -121.9144
CC.GTWY  46.7402  -121.9170
CC.KAUT  46.7303  -121.8574
CC.KAVK  46.7604  -122.0386
CC.LONE  47.0099  -121.6737
CC.LONR  47.0099  -121.6737
CC.MILD  46.8081  -121.7757
CC.OBSR  46.8997  -121.8153
CC.OPCH  46.7313  -121.5713
CC.PANH  46.8590  -121.6426
CC.PARA  46.7864  -121.7422
CC.PR01  46.9101  -122.0376
CC.PR02  46.9183  -122.0487
CC.PR03  46.9034  -122.0327
CC.PR04  46.9298  -121.9886
CC.PR05  46.8417  -121.9489
CC.RUSH  46.9031  -121.9442
CC.SIFT  46.8668  -121.9533
CC.TABR  46.8043  -121.8495
CC.TAVI  46.7957  -121.8842
CC.VOIT  46.9664  -121.9833
CC.WOW  46.7800  -121.8851
UW.FMW  46.9419  -121.6715
UW.LO2  46.7506  -121.8096
UW.LON  46.7506  -121.8096
UW.RCM  46.8356  -121.7332
UW.RCS  46.8708  -121.7323
UW.RER  46.8186  -121.8425
UW.STAR  46.8508  -121.7930

JSON:
[{"name": "CC.ARAT",

4858

In [ ]:
# daily_detection_dynamic.py
#
# Drop-in replacement for daily_detection.py that dynamically fetches the
# list of stations active on the requested date within a radius of Mt Rainier,
# instead of reading a fixed stations.json.
#
# Everything downstream (model inference, event detection, CSV output) is
# identical to the original script.
#
# Usage:
#   python daily_detection_dynamic.py \
#       --start 2025-12-10T00:00:00 \
#       --end   2025-12-10T23:59:59
#
# Optional overrides:
#   --networks   *              (default: all networks within radius)
#   --radius_km  50             (default: 50)
#   --lat        46.8529        (default: Mt Rainier summit)
#   --lon       -121.7604
#   --channels   BH,HH,EH      (comma-separated prefixes to accept)
#   --fdsn_client IRIS          (default: IRIS)
#   --min_duration_hours 1      (skip stations with <N hours of data that day)

import os
import csv
import json
import argparse
import numpy as np
import pandas as pd
import torch
import obspy
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

import sys
sys.path.append('/home/ak287/seisbench/seisbench/models')

import seisbench.models as sbm
from detect import smooth_moving_avg, detect_event_windows


# ── constants ─────────────────────────────────────────────────────────────────
RAINIER_LAT  = 46.8529
RAINIER_LON  = -121.7604
DEFAULT_RADIUS_KM = 50.0
# Approximate degrees per km at this latitude
KM_PER_DEG_LAT = 111.0
KM_PER_DEG_LON = 111.0 * 0.682   # cos(46.85°)

# Channel prefixes we trust QuakeXNet to handle well (broadband first)
PREFERRED_CHANNEL_PREFIXES = ["BH", "HH", "EH", "SH"]

# Networks to search
DEFAULT_NETWORKS = "*"    # all networks within radius


# ── argument parsing ──────────────────────────────────────────────────────────
parser = argparse.ArgumentParser(
    description="Run QuakeXNet detection with dynamically fetched station list."
)
parser.add_argument("--start", type=str, required=True,
                    help="Start time UTC (e.g. '2025-12-10T00:00:00')")
parser.add_argument("--end",   type=str, required=True,
                    help="End time UTC (e.g. '2025-12-10T23:59:59')")
parser.add_argument("--networks", type=str, default=DEFAULT_NETWORKS,
                    help="Network codes to search, comma-separated or * for all (default: *)")
parser.add_argument("--radius_km", type=float, default=DEFAULT_RADIUS_KM,
                    help=f"Search radius in km (default: {DEFAULT_RADIUS_KM})")
parser.add_argument("--lat", type=float, default=RAINIER_LAT,
                    help="Center latitude (default: Mt Rainier summit)")
parser.add_argument("--lon", type=float, default=RAINIER_LON,
                    help="Center longitude (default: Mt Rainier summit)")
parser.add_argument("--channels", type=str,
                    default=",".join(PREFERRED_CHANNEL_PREFIXES),
                    help="Comma-separated channel prefixes in preference order")
parser.add_argument("--fdsn_client", type=str, default="IRIS",
                    help="FDSN client name (default: IRIS)")
parser.add_argument("--min_duration_hours", type=float, default=1.0,
                    help="Skip stations with less than this many hours of "
                         "data in the requested window (default: 1)")
parser.add_argument("--save_station_list", action="store_true",
                    help="Save the dynamically built station list to a JSON "
                         "file alongside the detection output")
args = parser.parse_args()

st_time = UTCDateTime(args.start)
et_time = UTCDateTime(args.end)
networks = [n.strip() for n in args.networks.split(",")]
channel_prefixes = [c.strip() for c in args.channels.split(",")]
radius_deg = args.radius_km / KM_PER_DEG_LAT   # rough conversion for FDSN

print(f"\n{'='*60}")
print(f"QuakeXNet dynamic detection")
print(f"  Window : {st_time}  →  {et_time}")
print(f"  Center : {args.lat}°N  {args.lon}°E")
print(f"  Radius : {args.radius_km} km  (~{radius_deg:.3f}°)")
print(f"  Nets   : {networks if networks != ["*"] else "ALL"}")
print(f"{'='*60}\n")


# ── 1. dynamically fetch station list ─────────────────────────────────────────
def fetch_active_stations(client, lat, lon, radius_deg,
                          starttime, endtime, channel_prefixes,
                          min_duration_hours):
    """
    Query FDSN for all stations in the given networks within radius_deg of
    (lat, lon) that were active during [starttime, endtime].

    For each station, pick the best available channel prefix from
    channel_prefixes (in order of preference).

    Returns a list of dicts: [{"net": ..., "sta": ..., "chn": ...}, ...]
    """
    print("Fetching active station list from FDSN...")
    stations_out = []
    seen = set()   # (net, sta) dedup

    try:
        inv = client.get_stations(
            network="*",           # all networks
            latitude=lat,
            longitude=lon,
            maxradius=radius_deg,
            starttime=starttime,
            endtime=endtime,
            level="channel",
        )
    except Exception as e:
        print(f"  ERROR — FDSN query failed: {e}")
        return stations_out

    for net_obj in inv:
            for sta_obj in net_obj:

                # check station epoch overlaps the requested window
                sta_start = sta_obj.start_date
                sta_end   = sta_obj.end_date if sta_obj.end_date else \
                            UTCDateTime(2099, 1, 1)
                if sta_start > endtime or sta_end < starttime:
                    continue

                sta_key = (net_obj.code, sta_obj.code)
                if sta_key in seen:
                    continue

                # find the best channel prefix available at this station
                available_prefixes = set()
                for cha in sta_obj.channels:
                    for pfx in channel_prefixes:
                        if cha.code.startswith(pfx) and cha.code.endswith("Z"):
                            available_prefixes.add(pfx)

                best_prefix = None
                for pfx in channel_prefixes:   # respects preference order
                    if pfx in available_prefixes:
                        best_prefix = pfx
                        break

                if best_prefix is None:
                    print(f"  SKIP {net_obj.code}.{sta_obj.code} — "
                          f"no matching channel prefix in {channel_prefixes}")
                    continue

                seen.add(sta_key)
                stations_out.append({
                    "net": net_obj.code,
                    "sta": sta_obj.code,
                    "chn": best_prefix,
                })

    # sort for reproducibility: by net then sta
    stations_out.sort(key=lambda x: (x["net"], x["sta"]))

    print(f"  Found {len(stations_out)} active stations:")
    for s in stations_out:
        print(f"    {s['net']}.{s['sta']:8s}  [{s['chn']}*]")
    print()

    return stations_out


fdsn_client = Client(args.fdsn_client)

stations = fetch_active_stations(
    client          = fdsn_client,
    lat             = args.lat,
    lon             = args.lon,
    radius_deg      = radius_deg,
    starttime       = st_time,
    endtime         = et_time,
    channel_prefixes= channel_prefixes,
    min_duration_hours = args.min_duration_hours,
)

if not stations:
    print("ERROR: No active stations found for this date/region. Exiting.")
    raise SystemExit(1)


# ── 2. load model ─────────────────────────────────────────────────────────────
print("Loading QuakeXNet model...")
model = sbm.QuakeXNet.from_pretrained("base", version_str='3')
print("  Model loaded.\n")


# ── 3. set up output paths ────────────────────────────────────────────────────
os.makedirs("../plots", exist_ok=True)
os.makedirs("../logs",  exist_ok=True)

start_str = st_time.strftime("%Y%m%d_%H%M")
end_str   = et_time.strftime("%Y%m%d_%H%M")
out_dir   = f"../logs/mt_rainier_detections/{start_str}_{end_str}"
os.makedirs(out_dir, exist_ok=True)

# optionally save the station list used for this run
if args.save_station_list:
    sta_list_path = os.path.join(out_dir, f"stations_{start_str}.json")
    with open(sta_list_path, "w") as f:
        json.dump(stations, f, indent=2)
    print(f"  Station list saved → {sta_list_path}")

log_file = "../logs/detections.csv"
if not os.path.exists(log_file):
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["date", "station", "final_label",
                         "eq_auc", "px_auc", "su_auc"])


# ── 4. main detection loop (identical to original) ────────────────────────────
class_names  = ["eq", "px", "su"]
chn_prefix   = "QuakeXNet_"
channel_map  = {cls: f"{chn_prefix}{cls}" for cls in class_names}
SECONDS_PER_STEP = 10   # stride=500 samples @ 50 Hz → 10 s per step

print(f"{'='*60}")
print(f"Running detection on {len(stations)} stations...")
print(f"{'='*60}\n")

for entry in stations:
    net = entry["net"]
    sta = entry["sta"]
    chn = entry["chn"]

    print(f"🔍  Processing {net}.{sta}  [{chn}*]...")

    try:
        st = fdsn_client.get_waveforms(
            network   = net,
            station   = sta,
            channel   = chn + "*",
            location  = "*",
            starttime = st_time,
            endtime   = et_time,
        )

        print(f"  Raw stream: {st}")

        # deduplicate location codes — prefer "" then "00" then first available
        seen_channels = {}
        for tr in sorted(st, key=lambda t: t.stats.location):
            key = tr.stats.channel
            if key not in seen_channels:
                seen_channels[key] = tr
        st_clean = obspy.Stream(list(seen_channels.values()))

        # resample to 100 sps
        target_fs = 100.0
        for tr in st_clean:
            if tr.stats.sampling_rate != target_fs:
                print(f"  Resampling {tr.id}: "
                      f"{tr.stats.sampling_rate} → {target_fs} sps")
                tr.resample(target_fs)

        # merge to fill intra-day gaps
        st_clean.merge(method=1, fill_value=0)
        print(f"  Clean stream ({target_fs} sps): {st_clean}")

        # skip if we got almost no data
        if not st_clean or \
           max(tr.stats.npts for tr in st_clean) < \
           args.min_duration_hours * 3600 * target_fs:
            print(f"  SKIP — insufficient data "
                  f"(<{args.min_duration_hours}h)\n")
            continue

        # run inference
        probs_st = model.annotate(st_clean, stride=500)

        event_records = []

        for cls in class_names:
            probs = probs_st.select(channel=channel_map[cls])

            for prob in probs:
                trace_start  = prob.stats.starttime
                probs_array  = prob.data
                s_cls        = smooth_moving_avg(probs_array)
                events       = detect_event_windows(s_cls)

                for event in events:
                    start_idx = event["start"]
                    end_idx   = event["end"]

                    event_start_time = trace_start + start_idx * SECONDS_PER_STEP
                    event_end_time   = trace_start + end_idx   * SECONDS_PER_STEP

                    if event_end_time < st_time or event_start_time > et_time:
                        continue

                    event_records.append({
                        "station":     sta,
                        "network":     net,
                        "class":       cls,
                        "auc":         event["area_under_curve"],
                        "mean_prob":   event["mean_prob"],
                        "max_prob":    event["max_prob"],
                        "start_index": start_idx,
                        "end_index":   end_idx,
                        "start_time":  str(event_start_time),
                        "end_time":    str(event_end_time),
                    })

        if event_records:
            df_events = pd.DataFrame(event_records)
            print(df_events.to_string(index=False))

            out_csv = os.path.join(
                out_dir,
                f"{sta}_{start_str}_to_{end_str}_events.csv"
            )
            df_events.to_csv(out_csv, index=False)
            print(f"  Saved → {out_csv}")
        else:
            print(f"  No events detected.")

        print()

    except Exception as e:
        print(f"  ❌  Error processing {net}.{sta}: {e}\n")

print(f"\n{'='*60}")
print(f"Done. Results in: {out_dir}")
print(f"{'='*60}")

## How do I make location code faster? 

In [31]:
import pandas as pd
df1 = pd.read_csv("../src/catalog_output/located_events_checkpoint_1000.csv")
df2 = pd.read_csv("../src/catalog_output/located_events_checkpoint_2000.csv")
print(f"checkpoint_1000: {len(df1)} rows")
print(f"checkpoint_2000: {len(df2)} rows")
print(f"Last event_id in 1000: {df1['event_id'].iloc[-1]}")
print(f"Last event_id in 2000: {df2['event_id'].iloc[-1]}")

checkpoint_1000: 86 rows
checkpoint_2000: 149 rows
Last event_id in 1000: MR0001344
Last event_id in 2000: MR0002729


In [32]:
import pandas as pd
catalog = pd.read_csv("../src/catalog_output/master_catalog.csv")
catalog = catalog[catalog["most_common_class"].isin(["su", "px"])].copy().reset_index(drop=True)
print(f"Total su/px events: {len(catalog)}")
print(f"Max index: {catalog.index.max()}")

Total su/px events: 454739
Max index: 454738


## Comparing with PNSN catalog

In [58]:
import pandas as pd

pnsn = pd.read_csv("../data/px_su_2010_to_22_APR_2026.csv")
pnsn["orig_time_true"] = pd.to_datetime(pnsn["orig_time_true"], utc=True, format = 'mixed')

# one row per event
pnsn_events = pnsn.drop_duplicates(subset=["evid"]).copy()
pnsn_events = pnsn_events[["evid", "orig_time_true", "lat", "lon",
                            "depth", "etype", "totalarr"]].reset_index(drop=True)

print(f"Total PNSN events: {len(pnsn_events)}")
print(pnsn_events["etype"].value_counts())

Total PNSN events: 14754
etype
su    8022
px    6732
Name: count, dtype: int64


In [59]:
from obspy import UTCDateTime
from pnwstore import StationClient
import numpy as np
import pandas as pd

USERNAME = "cascadia"
PASSWORD = "pnwstore"

RAINIER_LAT  = 46.8529
RAINIER_LON  = -121.7604
THRESHOLD_KM = 150
MIN_FRACTION = 0.5

def dist_km(lat1, lon1, lat2, lon2):
    dlat = lat1 - lat2
    dlon = (lon1 - lon2) * np.cos(np.radians((lat1 + lat2) / 2))
    return np.sqrt((dlat * 111)**2 + (dlon * 111)**2)

# ── fetch coordinates for all PNSN stations via pnwstore ─────────────────────
station_client = StationClient(USERNAME, PASSWORD)

pnsn_stations = pnsn[["net", "sta"]].drop_duplicates().reset_index(drop=True)
print(f"Fetching coordinates for {len(pnsn_stations)} stations...")

coords_cache = {}
for i, row in pnsn_stations.iterrows():
    net, sta = row["net"], row["sta"]
    try:
        df = station_client.query(
            network=net,
            station=sta,
            mintime=UTCDateTime("2009-01-01"),
            maxtime=UTCDateTime("2026-12-31")
        )
        if df is not None and len(df) > 0:
            coords_cache[(net, sta)] = (
                df["latitude"].iloc[0],
                df["longitude"].iloc[0]
            )
        else:
            coords_cache[(net, sta)] = None
    except Exception as e:
        coords_cache[(net, sta)] = None

    if (i + 1) % 50 == 0:
        found_so_far = sum(1 for v in coords_cache.values() if v is not None)
        print(f"  {i+1}/{len(pnsn_stations)} done  (found: {found_so_far})")

found  = sum(1 for v in coords_cache.values() if v is not None)
missed = sum(1 for v in coords_cache.values() if v is None)
print(f"\nCoordinates found: {found}  |  Not found: {missed}")

# ── compute distance from Rainier ─────────────────────────────────────────────
station_dist = {}
for (net, sta), coords in coords_cache.items():
    if coords is None:
        station_dist[(net, sta)] = None
    else:
        station_dist[(net, sta)] = dist_km(
            coords[0], coords[1], RAINIER_LAT, RAINIER_LON
        )

# show distribution to help tune threshold
dists = pd.Series([v for v in station_dist.values() if v is not None])
print(f"\nStation distance from Rainier (km):")
print(f"  min={dists.min():.1f}  median={dists.median():.1f}  max={dists.max():.1f}")
print(f"  within  50km: {(dists <=  50).sum()}")
print(f"  within 100km: {(dists <= 100).sum()}")
print(f"  within 150km: {(dists <= 150).sum()}")
print(f"  within 200km: {(dists <= 200).sum()}")
print(f"  within 300km: {(dists <= 300).sum()}")

# ── tag each arrival with station distance ────────────────────────────────────
pnsn["sta_dist_km"] = pnsn.apply(
    lambda r: station_dist.get((r["net"], r["sta"])), axis=1
)

# ── filter events by station proximity ────────────────────────────────────────
def is_rainier_event(group):
    dists = group["sta_dist_km"].dropna()
    if len(dists) == 0:
        return False
    return (dists <= THRESHOLD_KM).mean() >= MIN_FRACTION

print(f"\nFiltering events (threshold={THRESHOLD_KM}km, "
      f"min_fraction={MIN_FRACTION})...")

rainier_mask = (
    pnsn.groupby("evid")
        .apply(is_rainier_event)
        .rename("is_rainier")
        .reset_index()
)

pnsn_rainier = (
    pnsn.merge(rainier_mask, on="evid")
        .query("is_rainier")
        .drop(columns="is_rainier")
)

pnsn_rainier_events = (
    pnsn_rainier
    .drop_duplicates(subset=["evid"])
    [["evid", "orig_time_true", "lat", "lon",
      "depth", "etype", "totalarr"]]
    .reset_index(drop=True)
)

print(f"\nPNSN events in full catalog:          {pnsn['evid'].nunique():,}")
print(f"PNSN Rainier events (<{THRESHOLD_KM}km):   {len(pnsn_rainier_events):,}")
print(f"\nEvent type breakdown:")
print(pnsn_rainier_events["etype"].value_counts())
print(f"\nEvents with location (lat!=0):   "
      f"{(pnsn_rainier_events['lat'] != 0).sum()}")
print(f"Events without location (lat=0): "
      f"{(pnsn_rainier_events['lat'] == 0).sum()}")

Fetching coordinates for 813 stations...
  50/813 done  (found: 17)
  100/813 done  (found: 29)
  150/813 done  (found: 42)
  200/813 done  (found: 57)
  250/813 done  (found: 63)
  300/813 done  (found: 65)
  350/813 done  (found: 68)
  400/813 done  (found: 68)
  450/813 done  (found: 69)
  500/813 done  (found: 69)
  550/813 done  (found: 69)
  600/813 done  (found: 69)
  650/813 done  (found: 69)
  700/813 done  (found: 69)
  750/813 done  (found: 69)
  800/813 done  (found: 69)

Coordinates found: 69  |  Not found: 744

Station distance from Rainier (km):
  min=2.9  median=172.0  max=559.9
  within  50km: 3
  within 100km: 18
  within 150km: 30
  within 200km: 45
  within 300km: 58

Filtering events (threshold=150km, min_fraction=0.5)...

PNSN events in full catalog:          14,754
PNSN Rainier events (<150km):   5,009

Event type breakdown:
etype
px    3121
su    1888
Name: count, dtype: int64

Events with location (lat!=0):   4817
Events without location (lat=0): 192


/tmp/ipykernel_665886/4274043666.py:89: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


In [60]:
# test one station that we know exists
df = station_client.query(
    network="UW",
    station="SHW",
    mintime=UTCDateTime("2009-01-01"),
    maxtime=UTCDateTime("2026-12-31")
)
print(df)
print(df.columns.tolist())
print(df.dtypes)

Empty DataFrame
Columns: [channel_id, network, station, location, channel, latitude, longitude, elevation, depth, starttime, endtime, sampling_rate, azimuth]
Index: []
['channel_id', 'network', 'station', 'location', 'channel', 'latitude', 'longitude', 'elevation', 'depth', 'starttime', 'endtime', 'sampling_rate', 'azimuth']
channel_id       object
network          object
station          object
location         object
channel          object
latitude         object
longitude        object
elevation        object
depth            object
starttime        object
endtime          object
sampling_rate    object
azimuth          object
dtype: object


In [61]:
# test a PB station that failed
df2 = station_client.query(
    network="PB",
    station="B202",
    mintime=UTCDateTime("2009-01-01"),
    maxtime=UTCDateTime("2026-12-31")
)
print(df2)

    channel_id network station location channel  latitude  longitude  \
0        15537      PB    B202       --     EH1   46.2447   -122.137   
1        15538      PB    B202       --     EH2   46.2447   -122.137   
2        15539      PB    B202       --     EHZ   46.2447   -122.137   
3        15540      PB    B202       --     LCC   46.2447   -122.137   
4        15541      PB    B202       --     LCE   46.2447   -122.137   
..         ...     ...     ...      ...     ...       ...        ...   
71       15608      PB    B202       T6     RCB   46.2447   -122.137   
72       15609      PB    B202       T6     RCC   46.2447   -122.137   
73       15610      PB    B202       T6     RCD   46.2447   -122.137   
74       15611      PB    B202       TS     RDO   46.2447   -122.137   
75       15612      PB    B202       TS     RRO   46.2447   -122.137   

   elevation   depth     starttime        endtime sampling_rate azimuth  
0    1218.60  236.80  1185321600.0  19880899200.0      100.00

In [62]:
# try querying by channel only to see what comes back
df3 = station_client.query(
    channel="EH?",
    mintime=UTCDateTime("2010-01-01"),
    maxtime=UTCDateTime("2010-01-02")
)
print(f"Rows returned: {len(df3)}")
print(df3.head())
print(df3.columns.tolist())

Rows returned: 468
   channel_id network station location channel  latitude  longitude elevation  \
0        6925      CC    CLMS       --     EHZ   42.9230   -122.016   2719.00   
1        7085      CC     NED       --     EHZ   46.2002   -122.186   2060.00   
2        7301      CC     SEP       --     EHE   46.2002   -122.191   2116.00   
3        7306      CC     SEP       --     EHN   46.2002   -122.191   2116.00   
4        7313      CC     SEP       --     EHZ   46.2002   -122.191   2116.00   

  depth     starttime       endtime sampling_rate azimuth  
0  0.00  1250640000.0  1373932800.0      100.0000    None  
1  0.00  1150761600.0  1367366400.0      100.0000    None  
2  0.00  1140739200.0  1367280000.0      100.0000   90.00  
3  0.00  1140739200.0  1367280000.0      100.0000    None  
4  0.00  1140739200.0  1367280000.0      100.0000    None  
['channel_id', 'network', 'station', 'location', 'channel', 'latitude', 'longitude', 'elevation', 'depth', 'starttime', 'endtime', 'sa

In [65]:
coords_cache = {}

print(f"Fetching coordinates for {len(pnsn_stations)} stations...")
for i, row in pnsn_stations.iterrows():
    net, sta = row["net"], row["sta"]
    try:
        df = station_client.query(
            network=net,
            station=sta,
            channel="*",  # wildcard — get all channels
            #mintime=UTCDateTime("2009-01-01"),
            #maxtime=UTCDateTime("2026-12-31")
        )
        if df is not None and len(df) > 0:
            coords_cache[(net, sta)] = (
                float(df["latitude"].iloc[0]),
                float(df["longitude"].iloc[0])
            )
        else:
            coords_cache[(net, sta)] = None
    except Exception as e:
        coords_cache[(net, sta)] = None

    if (i + 1) % 50 == 0:
        found_so_far = sum(1 for v in coords_cache.values() if v is not None)
        print(f"  {i+1}/{len(pnsn_stations)} done  (found: {found_so_far})")

found  = sum(1 for v in coords_cache.values() if v is not None)
missed = sum(1 for v in coords_cache.values() if v is None)
print(f"\nCoordinates found: {found}  |  Not found: {missed}")

Fetching coordinates for 813 stations...
  50/813 done  (found: 50)
  100/813 done  (found: 97)
  150/813 done  (found: 144)
  200/813 done  (found: 191)
  250/813 done  (found: 234)
  300/813 done  (found: 281)
  350/813 done  (found: 328)
  400/813 done  (found: 374)
  450/813 done  (found: 423)
  500/813 done  (found: 472)
  550/813 done  (found: 522)
  600/813 done  (found: 570)
  650/813 done  (found: 617)
  700/813 done  (found: 652)
  750/813 done  (found: 673)
  800/813 done  (found: 684)

Coordinates found: 686  |  Not found: 127


In [66]:
missing = [(net, sta) for (net, sta), v in coords_cache.items() if v is None]
missing_df = pd.DataFrame(missing, columns=["net", "sta"])

print(f"Missing stations: {len(missing_df)}")
print("\nBy network:")
print(missing_df["net"].value_counts())
print("\nFull list:")
print(missing_df.to_string(index=False))

Missing stations: 127

By network:
net
UW    38
UO    31
CC    15
NC    13
MB     8
BK     6
HW     5
WW     5
2M     3
IW     2
NN     1
Name: count, dtype: int64

Full list:
net   sta
 HW   GBB
 HW   SNI
 HW   BEN
 HW   H2O
 NC   KTR
 BK  HUMO
 HW  WIXX
 NC  KSXB
 MB  BSMT
 NC   KBO
 NC   KEB
 BK   YBH
 NC   LHE
 BK   MOD
 NC   LGB
 NC   LAS
 NC  LAMB
 NC   LBC
 NC   LTI
 NC   KRP
 MB  CHMT
 MB  JTMT
 MB  MKMT
 MB  BLMT
 NC   LMC
 NN  COLR
 BK  SHEP
 MB   LDM
 IW  MFID
 IW  PLID
 UW  PCOL
 UO  VAUN
 UW  OZET
 UW  NFES
 UW PSNS2
 UW  DART
 UW  CCFR
 UO CBARC
 UW TURTL
 UO ROCKA
 UO COOPR
 UO GROND
 UO  LOGZ
 UW  TOUT
 UO  JOBT
 UW  BLOB
 UO  BLEU
 UW  AGNW
 UW  MBW2
 UO HAMAK
 UO ASTOR
 UO  FIDL
 CC  ELBE
 CC  FLAT
 UO  VIP2
 UO  BUXT
 WW  TBID
 UO  SADL
 UW  HAWK
 UO BASIN
 UW GOLDN
 UO WALLO
 WW  BILL
 BK  SBAR
 UO CLOVR
 BK  CLRV
 UO  SLPT
 UW CHELN
 UW ROCKR
 UW ROCKI
 WW  CTNW
 UW BUCKS
 UW  ODUC
 UW  KALA
 UO  ADEL
 UW MAPLE
 UO PISTL
 UW OQNOB
 WW  IRMR
 UO   BPO
 UW  TEHA
 UO 

## The above stations were missed because they were deployed post 2022 for which PNWstore doesnt have data

In [68]:
from obspy.clients.fdsn import Client
import time

iris_client = Client("IRIS")
missing_df = pd.DataFrame(
    [(net, sta) for (net, sta), v in coords_cache.items() if v is None],
    columns=["net", "sta"]
)

print(f"Fetching {len(missing_df)} missing stations from IRIS...")
for i, row in missing_df.iterrows():
    net, sta = row["net"], row["sta"]
    try:
        inv = iris_client.get_stations(
            network=net, station=sta, level="station",
            #starttime=UTCDateTime("2009-01-01"),
            #endtime=UTCDateTime("2026-12-31")
        )
        lat, lon = None, None
        for net_obj in inv:
            for sta_obj in net_obj:
                lat = sta_obj.latitude
                lon = sta_obj.longitude
                break
            if lat is not None:
                break
        if lat is not None:
            coords_cache[(net, sta)] = (float(lat), float(lon))
    except Exception:
        pass

    #time.sleep(0.2)  # avoid rate limiting

found  = sum(1 for v in coords_cache.values() if v is not None)
missed = sum(1 for v in coords_cache.values() if v is None)
print(f"\nCoordinates found: {found}  |  Not found: {missed}")

# show any still missing
still_missing = [(net, sta) for (net, sta), v in coords_cache.items() if v is None]
if still_missing:
    print("\nStill missing:")
    for net, sta in still_missing:
        print(f"  {net}.{sta}")

Fetching 47 missing stations from IRIS...

Coordinates found: 808  |  Not found: 5

Still missing:
  HW.GBB
  HW.SNI
  HW.BEN
  HW.H2O
  HW.WIXX


In [69]:
# ── compute distance from Rainier for each station ────────────────────────────
station_dist = {}
for (net, sta), coords in coords_cache.items():
    if coords is None:
        station_dist[(net, sta)] = None
    else:
        station_dist[(net, sta)] = dist_km(
            coords[0], coords[1], RAINIER_LAT, RAINIER_LON
        )

# show distance distribution to tune threshold
dists = pd.Series([v for v in station_dist.values() if v is not None])
print(f"Station distance from Rainier (km):")
print(f"  min={dists.min():.1f}  median={dists.median():.1f}  max={dists.max():.1f}")
print(f"  within  50km: {(dists <=  50).sum()}")
print(f"  within 100km: {(dists <= 100).sum()}")
print(f"  within 150km: {(dists <= 150).sum()}")
print(f"  within 200km: {(dists <= 200).sum()}")
print(f"  within 300km: {(dists <= 300).sum()}")

# ── tag each arrival with station distance ────────────────────────────────────
pnsn["sta_dist_km"] = pnsn.apply(
    lambda r: station_dist.get((r["net"], r["sta"])), axis=1
)

# ── filter events by station proximity ────────────────────────────────────────
def is_rainier_event(group):
    dists = group["sta_dist_km"].dropna()
    if len(dists) == 0:
        return False
    return (dists <= THRESHOLD_KM).mean() >= MIN_FRACTION

print(f"\nFiltering events (threshold={THRESHOLD_KM}km, "
      f"min_fraction={MIN_FRACTION})...")

rainier_mask = (
    pnsn.groupby("evid")
        .apply(is_rainier_event)
        .rename("is_rainier")
        .reset_index()
)

pnsn_rainier = (
    pnsn.merge(rainier_mask, on="evid")
        .query("is_rainier")
        .drop(columns="is_rainier")
)

pnsn_rainier_events = (
    pnsn_rainier
    .drop_duplicates(subset=["evid"])
    [["evid", "orig_time_true", "lat", "lon",
      "depth", "etype", "totalarr"]]
    .reset_index(drop=True)
)

print(f"\nPNSN events in full catalog:             {pnsn['evid'].nunique():,}")
print(f"PNSN Rainier events (<{THRESHOLD_KM}km):      {len(pnsn_rainier_events):,}")
print(f"\nEvent type breakdown:")
print(pnsn_rainier_events["etype"].value_counts())
print(f"\nEvents with location (lat!=0):   "
      f"{(pnsn_rainier_events['lat'] != 0).sum()}")
print(f"Events without location (lat=0): "
      f"{(pnsn_rainier_events['lat'] == 0).sum()}")

Station distance from Rainier (km):
  min=2.5  median=181.7  max=668.6
  within  50km: 56
  within 100km: 168
  within 150km: 291
  within 200km: 459
  within 300km: 596

Filtering events (threshold=150km, min_fraction=0.5)...

PNSN events in full catalog:             14,754
PNSN Rainier events (<150km):      9,844

Event type breakdown:
etype
su    6669
px    3175
Name: count, dtype: int64

Events with location (lat!=0):   9598
Events without location (lat=0): 246


/tmp/ipykernel_665886/3000554569.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


In [70]:
for threshold in [50, 75, 100, 125, 150, 175, 200]:
    def is_rainier_event_t(group, t=threshold):
        dists = group["sta_dist_km"].dropna()
        if len(dists) == 0:
            return False
        return (dists <= t).mean() >= MIN_FRACTION

    mask = (
        pnsn.groupby("evid")
            .apply(is_rainier_event_t)
    )
    n = mask.sum()
    su = pnsn[pnsn["evid"].isin(mask[mask].index)].drop_duplicates("evid")["etype"].eq("su").sum()
    px = pnsn[pnsn["evid"].isin(mask[mask].index)].drop_duplicates("evid")["etype"].eq("px").sum()
    print(f"threshold={threshold:3d}km: {n:5d} events  (su={su}, px={px})")

/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


threshold= 50km:  3970 events  (su=3269, px=701)


/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


threshold= 75km:  4136 events  (su=3293, px=843)


/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


...

/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


threshold=150km:  9844 events  (su=6669, px=3175)


/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


threshold=175km: 10790 events  (su=7114, px=3676)
threshold=200km: 11560 events  (su=7209, px=4351)


/tmp/ipykernel_665886/3998184770.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


In [75]:
# ── set threshold to 100km ────────────────────────────────────────────────────
THRESHOLD_KM = 50
MIN_FRACTION = 0.5

def is_rainier_event(group):
    dists = group["sta_dist_km"].dropna()
    if len(dists) == 0:
        return False
    return (dists <= THRESHOLD_KM).mean() >= MIN_FRACTION

rainier_mask = (
    pnsn.groupby("evid", group_keys=False)
        .apply(is_rainier_event)
        .rename("is_rainier")
        .reset_index()
)

pnsn_rainier = (
    pnsn.merge(rainier_mask, on="evid")
        .query("is_rainier")
        .drop(columns="is_rainier")
)

pnsn_rainier_events = (
    pnsn_rainier
    .drop_duplicates(subset=["evid"])
    [["evid", "orig_time_true", "lat", "lon",
      "depth", "etype", "totalarr"]]
    .reset_index(drop=True)
)

print(f"PNSN Rainier events (<{THRESHOLD_KM}km): {len(pnsn_rainier_events):,}")
print(pnsn_rainier_events["etype"].value_counts())

# ── match against QuakeXNet catalog ──────────────────────────────────────────
master = pd.read_csv("../src/catalog_output/master_catalog.csv")
master["rounded_start"] = pd.to_datetime(master["rounded_start"], utc=True, format = 'mixed')

WINDOW_SEC = 60

print(f"\nMatching {len(pnsn_rainier_events):,} PNSN events against "
      f"{len(master):,} QuakeXNet events...")

matches = []
for _, pnsn_row in pnsn_rainier_events.iterrows():
    t    = pnsn_row["orig_time_true"]
    diff = (master["rounded_start"] - t).abs()
    within = master[diff <= pd.Timedelta(seconds=WINDOW_SEC)]

    if len(within) > 0:
        best_idx = diff[within.index].idxmin()
        best     = master.loc[best_idx]
        matches.append({
            "evid":            pnsn_row["evid"],
            "pnsn_time":       t,
            "pnsn_etype":      pnsn_row["etype"],
            "pnsn_lat":        pnsn_row["lat"],
            "pnsn_lon":        pnsn_row["lon"],
            "event_id":        best["event_id"],
            "qxn_time":        best["rounded_start"],
            "qxn_class":       best["most_common_class"],
            "qxn_vote_margin": best["vote_margin"],
            "qxn_mean_max":    best["mean_max"],
            "time_diff_sec":   diff[best_idx].total_seconds(),
            "matched":         True,
        })
    else:
        matches.append({
            "evid":            pnsn_row["evid"],
            "pnsn_time":       t,
            "pnsn_etype":      pnsn_row["etype"],
            "pnsn_lat":        pnsn_row["lat"],
            "pnsn_lon":        pnsn_row["lon"],
            "event_id":        None,
            "qxn_time":        None,
            "qxn_class":       None,
            "qxn_vote_margin": None,
            "qxn_mean_max":    None,
            "time_diff_sec":   None,
            "matched":         False,
        })

matches_df = pd.DataFrame(matches)
matched   = matches_df[matches_df["matched"]]
unmatched = matches_df[~matches_df["matched"]]

# ── results ───────────────────────────────────────────────────────────────────
print(f"\n=== MATCHING RESULTS ===")
print(f"PNSN events matched:  {len(matched):,}  "
      f"({100*len(matched)/len(pnsn_rainier_events):.1f}%)")
print(f"PNSN events missed:   {len(unmatched):,}  "
      f"({100*len(unmatched)/len(pnsn_rainier_events):.1f}%)")

print(f"\n=== RECALL BY CLASS ===")
for etype in ["su", "px"]:
    total     = (pnsn_rainier_events["etype"] == etype).sum()
    matched_n = matched[matched["pnsn_etype"] == etype].shape[0]
    print(f"  {etype}: {matched_n}/{total} = {100*matched_n/total:.1f}% recall")

print(f"\n=== CONFUSION MATRIX (PNSN etype vs QuakeXNet class) ===")
confusion = pd.crosstab(
    matched["pnsn_etype"],
    matched["qxn_class"],
    margins=True
)
print(confusion)

print(f"\n=== TIME DIFFERENCE DISTRIBUTION (matched events) ===")
print(matched["time_diff_sec"].describe().round(2))

print(f"\n=== NOVEL DETECTIONS (QuakeXNet only, not in PNSN) ===")
matched_qxn_ids = set(matched["event_id"].dropna())
novel = master[~master["event_id"].isin(matched_qxn_ids)]
print(f"Total novel detections: {len(novel):,}")
print(novel["most_common_class"].value_counts())

print(f"\n=== RECALL BY YEAR ===")
matched["year"] = pd.to_datetime(matched["pnsn_time"]).dt.year
pnsn_rainier_events["year"] = pd.to_datetime(
    pnsn_rainier_events["orig_time_true"]).dt.year

recall_by_year = []
for year in sorted(pnsn_rainier_events["year"].unique()):
    total     = (pnsn_rainier_events["year"] == year).sum()
    matched_n = (matched["year"] == year).sum()
    recall_by_year.append({
        "year":       year,
        "total_pnsn": total,
        "matched":    matched_n,
        "recall_pct": round(100 * matched_n / total, 1) if total > 0 else 0,
    })

recall_df = pd.DataFrame(recall_by_year)
print(recall_df.to_string(index=False))

/tmp/ipykernel_665886/70172783.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid", group_keys=False)


PNSN Rainier events (<50km): 3,970
etype
su    3269
px     701
Name: count, dtype: int64

Matching 3,970 PNSN events against 656,973 QuakeXNet events...

=== MATCHING RESULTS ===
PNSN events matched:  2,611  (65.8%)
PNSN events missed:   1,359  (34.2%)

=== RECALL BY CLASS ===
  su: 2015/3269 = 61.6% recall
  px: 596/701 = 85.0% recall

=== CONFUSION MATRIX (PNSN etype vs QuakeXNet class) ===
qxn_class   eq   px    su   All
pnsn_etype                     
px          18  434   144   596
su          66  306  1643  2015
All         84  740  1787  2611

=== TIME DIFFERENCE DISTRIBUTION (matched events) ===
count    2611.00
mean       38.97
std        17.98
min         0.00
25%        23.23
50%        46.82
75%        53.34
max        59.98
Name: time_diff_sec, dtype: float64

=== NOVEL DETECTIONS (QuakeXNet only, not in PNSN) ===
Total novel detections: 654,364
most_common_class
su    397592
eq    202150
px     54622
Name: count, dtype: int64

=== RECALL BY YEAR ===
 year  total_pnsn  mat

/tmp/ipykernel_665886/70172783.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched["year"] = pd.to_datetime(matched["pnsn_time"]).dt.year


In [76]:
unmatched

,evid,pnsn_time,pnsn_etype,pnsn_lat,pnsn_lon,event_id,qxn_time,qxn_class,qxn_vote_margin,qxn_mean_max,time_diff_sec,matched
1,10788193,2010-01-14 23:07:01.360000+00:00,px,46.961000,-122.074833,None,NaT,None,NaN,NaN,NaN,False
2,10790463,2010-01-22 20:38:54.870000+00:00,px,46.610833,-122.249500,None,NaT,None,NaN,NaN,NaN,False
3,10783083,2010-02-03 21:54:25.590000+00:00,px,46.984167,-122.208667,None,NaT,None,NaN,NaN,NaN,False
8,10799308,2010-04-29 18:21:39.770000+00:00,px,46.922500,-122.113667,None,NaT,None,NaN,NaN,NaN,False
10,10802218,2010-05-10 20:45:26.430000+00:00,px,46.890667,-122.285167,None,NaT,None,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
3965,62244761,2026-04-20 01:25:40.429406+00:00,su,46.835600,-121.733000,None,NaT,None,NaN,NaN,NaN,False
3966,62244821,2026-04-20 13:18:48.912598+00:00,su,46.835600,-121.733000,None,NaT,None,NaN,NaN,NaN,False
3967,62245671,2026-04-22 15:20:23.662448+00:00,su,46.835600,-121.733000,None,NaT,None,NaN,NaN,NaN,False
3968,62246211,2026-04-22 19:51:48.189999+00:00,su,46.834167,-121.732333,None,NaT,None,NaN,NaN,NaN,False


In [98]:
pnsn[pnsn['evid'] == 10802218]

,evid,prefor,version,etype,arid,orig_time_true,lat,lon,depth,totalarr,nbs,erhor,sdep,orig_commit_time,net,sta,location,seedchan,iphase,quality,traveltime,sta_dist_km
887,10802218,1086638,3,px,10200008,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,GHW,NaN,EHZ,P,0.8,3.211,44.192026
888,10802218,1086638,3,px,10200083,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,PCMD,NaN,ENZ,P,1.0,0.670,41.218386
889,10802218,1086638,3,px,10200078,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,CC,PANH,NaN,BHZ,P,0.5,7.890,8.937013
890,10802218,1086638,3,px,10200073,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,KOS,NaN,EHZ,P,0.8,8.081,54.555616
891,10802218,1086638,3,px,10200068,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,PB,B203,NaN,EHZ,P,0.5,12.898,87.651717
892,10802218,1086638,3,px,10200063,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,RCM,NaN,EHZ,S,0.8,12.751,2.831090
893,10802218,1086638,3,px,10200058,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,RCM,NaN,EHZ,P,0.8,7.091,2.831090
894,10802218,1086638,3,px,10200053,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,LO2,NaN,EHZ,P,0.8,6.391,11.964367
895,10802218,1086638,3,px,10200048,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,UW,LMW,NaN,EHZ,P,0.5,4.341,45.412927
896,10802218,1086638,3,px,10200043,2010-05-10 20:45:26.430000+00:00,46.890667,-122.285167,4.349,18.0,3.0,0.042,0.06,2012-03-07 19:52:59,PB,B941,NaN,EHZ,P,0.8,2.078,37.812460


In [93]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from obspy import UTCDateTime, Stream
from pnwstore import WaveformClient
from obspy.clients.fdsn import Client as FDSNClient

# ── event info ────────────────────────────────────────────────────────────────
ORIGIN_TIME = UTCDateTime("2010-01-14T23:07:01.36")
SECONDS_BEFORE = 10
SECONDS_AFTER  = 120
T_START = ORIGIN_TIME - SECONDS_BEFORE
T_END   = ORIGIN_TIME + SECONDS_AFTER

# stations from PNSN arrivals with their travel times
stations = [
    {"net": "UW", "sta": "RER",  "chan": "EHZ", "traveltime": 3.61},
    {"net": "UW", "sta": "LO2",  "chan": "EHZ", "traveltime": 5.21},
    {"net": "PB", "sta": "B018", "chan": "EHZ", "traveltime": 12.40},
    {"net": "UW", "sta": "GHW",  "chan": "EHZ", "traveltime": 3.35},
    {"net": "UW", "sta": "RCM",  "chan": "EHZ", "traveltime": 5.22},
    {"net": "UW", "sta": "STAR", "chan": "EHZ", "traveltime": 4.48},
    {"net": "PB", "sta": "B941", "chan": "EHZ", "traveltime": 1.77},
]

pnw_client  = WaveformClient()
fdsn_client = FDSNClient("IRIS")

# ── download waveforms ────────────────────────────────────────────────────────
stream = Stream()
status = {}

for s in stations:
    net, sta, chan = s["net"], s["sta"], s["chan"]
    label = f"{net}.{sta}.{chan}"
    got_data = False

    # try pnwstore first
    try:
        st = pnw_client.get_waveforms(
            network=net, station=sta,
            location="*", channel=chan,
            starttime=T_START, endtime=T_END
        )
        if st:
            stream += st
            status[label] = "pnwstore ✓"
            got_data = True
    except Exception:
        pass

    # fallback to FDSN
    if not got_data:
        try:
            st = fdsn_client.get_waveforms(
                net, sta, "*", chan, T_START, T_END
            )
            if st:
                stream += st
                status[label] = "FDSN ✓"
                got_data = True
        except Exception:
            pass

    if not got_data:
        status[label] = "no data ✗"

print("Data retrieval status:")
for k, v in status.items():
    print(f"  {k}: {v}")
print(f"\nTotal traces: {len(stream)}")

# ── plot ──────────────────────────────────────────────────────────────────────
if len(stream) == 0:
    print("No data to plot.")
else:
    stream.detrend("demean")
    stream.taper(max_percentage=None, max_length=5)
    stream.filter("bandpass", freqmin=1.0, freqmax=15.0,
                  corners=3, zerophase=True)

    n_traces = len(stream)
    fig, axes = plt.subplots(n_traces, 1,
                             figsize=(14, 2.5 * n_traces),
                             sharex=True)
    if n_traces == 1:
        axes = [axes]

    for ax, tr in zip(axes, stream):
        # time axis relative to origin
        times = tr.times(reftime=ORIGIN_TIME)
        ax.plot(times, tr.data, color="black", lw=0.6)

        # mark predicted arrival
        label_key = f"{tr.stats.network}.{tr.stats.station}.{tr.stats.channel}"
        tt = next((s["traveltime"] for s in stations
                   if s["sta"] == tr.stats.station), None)
        if tt is not None:
            ax.axvline(tt, color="red", lw=1.2, ls="--",
                       label=f"P arrival ({tt:.2f}s)")
            ax.legend(fontsize=7, loc="upper right")

        ax.axvline(0, color="blue", lw=1.0, ls=":",
                   alpha=0.7)  # origin time

        src = status.get(label_key, "")
        ax.set_ylabel(f"{tr.stats.network}.{tr.stats.station}\n"
                      f"{tr.stats.channel}  [{src}]",
                      fontsize=8, rotation=0,
                      labelpad=80, va="center")
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel("Time relative to origin (s)", fontsize=10)
    axes[0].set_title(
        f"PNSN event 10788193 — px — 2010-01-14 23:07:01 UTC\n"
        f"lat=46.961  lon=-122.075  depth=-1.29km",
        fontsize=11, fontweight="bold"
    )

    plt.tight_layout()
    outpath = "../src/catalog_output/event_10788193_waveforms.png"
    plt.savefig(outpath, dpi=150, bbox_inches="tight",
                facecolor="white")
    plt.close()
    print(f"\nSaved → {outpath}")

PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


Data retrieval status:
  UW.RER.EHZ: pnwstore ✓
  UW.LO2.EHZ: pnwstore ✓
  PB.B018.EHZ: pnwstore ✓
  UW.GHW.EHZ: pnwstore ✓
  UW.RCM.EHZ: pnwstore ✓
  UW.STAR.EHZ: no data ✗
  PB.B941.EHZ: pnwstore ✓

Total traces: 6

Saved → ../src/catalog_output/event_10788193_waveforms.png


In [95]:
from obspy import UTCDateTime
from pnwstore import WaveformClient

client = WaveformClient()
fdsn_client = Client('IRIS')

t_start = UTCDateTime("2010-01-14T00:00:00")
t_end   = UTCDateTime("2010-01-14T23:10:00")

for channel in ["EHZ", "EHN", "EHE", "EH1", "EH2"]:
    try:
        st = client.get_waveforms(
            network="UW",
            station="STAR",
            location="*",
            channel= "*",
            starttime=t_start,
            endtime=t_end
        )
        if st:
            print(f"UW.STAR.{channel}: {st}")
        else:
            print(f"UW.STAR.{channel}: empty stream returned")
    except Exception as e:
        print(f"UW.STAR.{channel}: ERROR — {type(e).__name__}: {e}")

PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


UW.STAR.EHZ: empty stream returned
UW.STAR.EHN: empty stream returned
UW.STAR.EHE: empty stream returned
UW.STAR.EH1: empty stream returned
UW.STAR.EH2: empty stream returned


In [57]:
from obspy import UTCDateTime
from pnwstore import StationClient
USERNAME = "cascadia"
PASSWORD = "pnwstore"

client = StationClient(USERNAME, PASSWORD)
client.query(network = "UW", channel = "EH?",
             mintime = UTCDateTime("2010-01-15"),
             maxtime = UTCDateTime("2000-01-16"))


PNWstore | INFO | package: mysql.connector.plugins
PNWstore | INFO | plugin_name: mysql_native_password
PNWstore | INFO | AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin


,channel_id,network,station,location,channel,latitude,longitude,elevation,depth,starttime,endtime,sampling_rate,azimuth
0,36914,UW,ALKI,--,EHZ,47.5751,-122.418,1.00,0.00,1099526400.0,1331251200.0,100.0000,None
1,37040,UW,ASR,--,EHZ,46.1526,-121.602,1357.00,0.00,790473600.0,1060300800.0,100.0000,None
2,37041,UW,ASR,--,EHZ,46.1526,-121.602,1357.00,0.00,1060300800.0,1207008000.0,100.0000,None
3,37042,UW,ASR,--,EHZ,46.1526,-121.602,1357.00,0.00,1207008000.0,1538524800.0,100.0000,None
4,37050,UW,ATES,--,EHZ,48.2362,-122.060,62.60,0.00,1046908800.0,1330041600.0,100.0000,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
482,71458,UW,HSR,--,EHZ,46.1743,-122.181,1720.00,0.00,1207008000.0,19880899200.0,100.0000,None
483,71580,UW,OD2,--,EHZ,47.3875,-118.711,553.00,0.00,790473600.0,1207008000.0,100.0000,None
484,71581,UW,OD2,--,EHZ,47.3875,-118.711,553.00,0.00,1207008000.0,19880899200.0,100.0000,None
485,71708,UW,VFP,--,EHZ,45.3179,-121.466,1716.00,0.00,867715200.0,1207008000.0,100.0000,None


In [38]:
master = pd.read_csv("../src/catalog_output/master_catalog.csv")
master["rounded_start"] = pd.to_datetime(master["rounded_start"], utc=True, format = 'mixed')

WINDOW_SEC = 60  # match within 60 seconds

matches = []
for _, pnsn_row in pnsn_events.iterrows():
    t = pnsn_row["orig_time_true"]
    # find catalog events within window
    diff = (master["rounded_start"] - t).abs()
    within = master[diff <= pd.Timedelta(seconds=WINDOW_SEC)]
    if len(within) > 0:
        best = within.loc[diff[within.index].idxmin()]
        matches.append({
            "evid":              pnsn_row["evid"],
            "pnsn_time":         t,
            "pnsn_etype":        pnsn_row["etype"],
            "pnsn_lat":          pnsn_row["lat"],
            "pnsn_lon":          pnsn_row["lon"],
            "event_id":          best["event_id"],
            "qxn_time":          best["rounded_start"],
            "qxn_class":         best["most_common_class"],
            "time_diff_sec":     diff[best.name].total_seconds(),
        })
    else:
        matches.append({
            "evid":         pnsn_row["evid"],
            "pnsn_time":    t,
            "pnsn_etype":   pnsn_row["etype"],
            "pnsn_lat":     pnsn_row["lat"],
            "pnsn_lon":     pnsn_row["lon"],
            "event_id":     None,
            "qxn_time":     None,
            "qxn_class":    None,
            "time_diff_sec": None,
        })

matches_df = pd.DataFrame(matches)
matched   = matches_df[matches_df["event_id"].notna()]
unmatched = matches_df[matches_df["event_id"].isna()]

print(f"\nPNSN events:   {len(pnsn_events)}")
print(f"Matched:       {len(matched)}  ({100*len(matched)/len(pnsn_events):.1f}%)")
print(f"Missed:        {len(unmatched)}  ({100*len(unmatched)/len(pnsn_events):.1f}%)")


PNSN events:   14754
Matched:       6514  (44.2%)
Missed:        8240  (55.8%)


In [39]:
# 1. How well does QuakeXNet classify PNSN-confirmed events?
print("\nQuakeXNet class for PNSN-confirmed su events:")
su_matched = matched[matched["pnsn_etype"] == "su"]
print(su_matched["qxn_class"].value_counts(normalize=True).round(3))

print("\nQuakeXNet class for PNSN-confirmed px events:")
px_matched = matched[matched["pnsn_etype"] == "px"]
print(px_matched["qxn_class"].value_counts(normalize=True).round(3))

# 2. What fraction of PNSN events did QuakeXNet detect at all?
print(f"\nRecall (su): {len(su_matched)/len(pnsn_events[pnsn_events['etype']=='su']):.1%}")
print(f"Recall (px): {len(px_matched)/len(pnsn_events[pnsn_events['etype']=='px']):.1%}")

# 3. What's in your catalog that PNSN missed?
# These are your novel detections
all_matched_ids = set(matched["event_id"])
novel = master[~master["event_id"].isin(all_matched_ids)]
print(f"\nNovel detections (not in PNSN): {len(novel):,}")
print(novel["most_common_class"].value_counts())


QuakeXNet class for PNSN-confirmed su events:
qxn_class
su    0.765
eq    0.118
px    0.117
Name: proportion, dtype: float64

QuakeXNet class for PNSN-confirmed px events:
qxn_class
px    0.485
su    0.436
eq    0.079
Name: proportion, dtype: float64

Recall (su): 41.8%
Recall (px): 47.0%

Novel detections (not in PNSN): 650,478
most_common_class
su    395446
eq    201590
px     53442
Name: count, dtype: int64


In [40]:
pnsn_stations = pnsn[["net", "sta"]].drop_duplicates().reset_index(drop=True)
print(f"Unique net/sta combinations: {len(pnsn_stations)}")
print(pnsn_stations.head(20))

Unique net/sta combinations: 813
   net   sta
0   PB  B202
1   UW   SHW
2   CC  VALT
3   UW   SEP
4   UW   HSR
5   CC   NED
6   UW   VLL
7   UW   TDH
8   CC  TIMB
9   UW  HOOD
10  UW   VFP
11  UW   SOS
12  CC   SUG
13  UW   STD
14  UW   EDM
15  UW   BKC
16  UW  BLOW
17  UW   BRO
18  UW   VIP
19  PB  B028


In [43]:
import numpy as np
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

# ── Rainier summit coordinates ────────────────────────────────────────────────
RAINIER_LAT  = 46.8529
RAINIER_LON  = -121.7604
THRESHOLD_KM = 150
MIN_FRACTION = 0.5

# ── load PNSN catalog ─────────────────────────────────────────────────────────
pnsn = pd.read_csv("../data/px_su_2010_to_22_APR_2026.csv")
pnsn["orig_time_true"] = pd.to_datetime(pnsn["orig_time_true"], utc=True, format="mixed")

print(f"Total PNSN rows (arrivals): {len(pnsn):,}")
print(f"Unique PNSN events:         {pnsn['evid'].nunique():,}")
print(f"Event types:\n{pnsn.drop_duplicates('evid')['etype'].value_counts()}\n")

# ── distance helper ───────────────────────────────────────────────────────────
def dist_km(lat1, lon1, lat2, lon2):
    dlat = lat1 - lat2
    dlon = (lon1 - lon2) * np.cos(np.radians((lat1 + lat2) / 2))
    return np.sqrt((dlat * 111)**2 + (dlon * 111)**2)

# ── fetch station coordinates ─────────────────────────────────────────────────
client        = Client("IRIS")
pnsn_stations = pnsn[["net", "sta"]].drop_duplicates().reset_index(drop=True)
coords_cache  = {}

print(f"Fetching coordinates for {len(pnsn_stations)} stations...")
for i, row in pnsn_stations.iterrows():
    net, sta = row["net"], row["sta"]
    try:
        inv = client.get_stations(
            network=net, station=sta, level="station",
            starttime=UTCDateTime("2009-01-01"),
            endtime=UTCDateTime("2026-12-31")
        )
        s = inv[0][0]
        coords_cache[(net, sta)] = (s.latitude, s.longitude)
    except Exception:
        coords_cache[(net, sta)] = None

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(pnsn_stations)} done...")

found  = sum(1 for v in coords_cache.values() if v is not None)
missed = sum(1 for v in coords_cache.values() if v is None)
print(f"\nCoordinates found: {found}  |  Not found: {missed}")

# ── compute distance from Rainier for each station ────────────────────────────
station_dist = {}
for (net, sta), coords in coords_cache.items():
    if coords is None:
        station_dist[(net, sta)] = None
    else:
        station_dist[(net, sta)] = dist_km(
            coords[0], coords[1], RAINIER_LAT, RAINIER_LON
        )

# ── tag each arrival row with its station distance ────────────────────────────
pnsn["sta_dist_km"] = pnsn.apply(
    lambda r: station_dist.get((r["net"], r["sta"])), axis=1
)

# show distance distribution for found stations
dists = pd.Series([v for v in station_dist.values() if v is not None])
print(f"\nStation distance from Rainier (km):")
print(f"  min={dists.min():.1f}  median={dists.median():.1f}  "
      f"max={dists.max():.1f}")
print(f"  within 50km:  {(dists <= 50).sum()}")
print(f"  within 100km: {(dists <= 100).sum()}")
print(f"  within 150km: {(dists <= 150).sum()}")
print(f"  within 200km: {(dists <= 200).sum()}")

# ── filter events by station proximity ────────────────────────────────────────
def is_rainier_event(group):
    dists = group["sta_dist_km"].dropna()
    if len(dists) == 0:
        return False
    return (dists <= THRESHOLD_KM).mean() >= MIN_FRACTION

print(f"\nFiltering events (threshold={THRESHOLD_KM}km, "
      f"min_fraction={MIN_FRACTION})...")

rainier_mask = (
    pnsn.groupby("evid")
        .apply(is_rainier_event)
        .rename("is_rainier")
        .reset_index()
)

pnsn_rainier = (
    pnsn.merge(rainier_mask, on="evid")
        .query("is_rainier")
        .drop(columns="is_rainier")
)

# one row per event
pnsn_rainier_events = (
    pnsn_rainier
    .drop_duplicates(subset=["evid"])
    [["evid", "orig_time_true", "lat", "lon",
      "depth", "etype", "totalarr"]]
    .reset_index(drop=True)
)

print(f"\nPNSN events in full catalog:         {pnsn['evid'].nunique():,}")
print(f"PNSN Rainier events (<{THRESHOLD_KM}km): {len(pnsn_rainier_events):,}")
print(f"\nEvent type breakdown:")
print(pnsn_rainier_events["etype"].value_counts())
print(f"\nEvents with location (lat!=0):  "
      f"{(pnsn_rainier_events['lat'] != 0).sum()}")
print(f"Events without location (lat=0): "
      f"{(pnsn_rainier_events['lat'] == 0).sum()}")

Total PNSN rows (arrivals): 103,439
Unique PNSN events:         14,754
Event types:
etype
su    8022
px    6732
Name: count, dtype: int64

Fetching coordinates for 813 stations...
  50/813 done...
  100/813 done...
  150/813 done...
  200/813 done...
  250/813 done...
  300/813 done...
  350/813 done...
  400/813 done...
  450/813 done...
  500/813 done...
  550/813 done...
  600/813 done...
  650/813 done...
  700/813 done...
  750/813 done...
  800/813 done...

Coordinates found: 0  |  Not found: 813

Station distance from Rainier (km):
  min=nan  median=nan  max=nan
  within 50km:  0
  within 100km: 0
  within 150km: 0
  within 200km: 0

Filtering events (threshold=150km, min_fraction=0.5)...

PNSN events in full catalog:         14,754
PNSN Rainier events (<150km): 0

Event type breakdown:
Series([], Name: count, dtype: int64)

Events with location (lat!=0):  0
Events without location (lat=0): 0


/tmp/ipykernel_665886/2964538304.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pnsn.groupby("evid")


Fetching coordinates for 813 stations...
  50/813 done  (found so far: 0)
  100/813 done  (found so far: 0)
  150/813 done  (found so far: 0)
  200/813 done  (found so far: 0)
  250/813 done  (found so far: 0)
  300/813 done  (found so far: 0)
  350/813 done  (found so far: 0)
  400/813 done  (found so far: 0)
  450/813 done  (found so far: 0)
  500/813 done  (found so far: 0)
  550/813 done  (found so far: 0)
  600/813 done  (found so far: 0)
  650/813 done  (found so far: 0)
  700/813 done  (found so far: 0)
  750/813 done  (found so far: 0)
  800/813 done  (found so far: 0)

Coordinates found: 0  |  Not found: 813


In [51]:
import json, glob, re
from pathlib import Path

# load all unique stations from your daily JSON files
station_json_dir = Path("../logs/mt_rainier_detections")
files = sorted(glob.glob(str(station_json_dir / "*" / "stations_*.json")))

your_stations = set()
for fpath in files:
    with open(fpath) as f:
        data = json.load(f)
    for entry in data:
        if entry.get("net") != "SY":
            your_stations.add(entry["sta"])

print(f"Unique stations in your network: {len(your_stations)}")
print(sorted(your_stations))

Unique stations in your network: 42
['ARAT', 'CARB', 'CLEV', 'COPP', 'CRBN', 'CRYS', 'CS1', 'CS2', 'CS3', 'CS4', 'FMW', 'GNOB', 'GOBB', 'GTWY', 'KAUT', 'KAVK', 'LO2', 'LON', 'LONE', 'LONR', 'MILD', 'OBSR', 'OPCH', 'PANH', 'PARA', 'PR01', 'PR02', 'PR03', 'PR04', 'PR05', 'RCM', 'RCS', 'RER', 'RUSH', 'RVC', 'SIFT', 'SR41', 'STAR', 'TABR', 'TAVI', 'VOIT', 'WOW']


In [52]:
# for each PNSN event, check if any of YOUR stations detected it
pnsn["detected_by_rainier_sta"] = pnsn["sta"].isin(your_stations)

rainier_event_ids = (
    pnsn.groupby("evid")["detected_by_rainier_sta"]
        .any()
        .reset_index()
        .query("detected_by_rainier_sta")["evid"]
)

pnsn_rainier = pnsn[pnsn["evid"].isin(rainier_event_ids)].copy()

pnsn_rainier_events = (
    pnsn_rainier
    .drop_duplicates(subset=["evid"])
    [["evid", "orig_time_true", "lat", "lon", "depth", "etype", "totalarr"]]
    .reset_index(drop=True)
)

print(f"\nPNSN events detected by at least one Rainier station: {len(pnsn_rainier_events)}")
print(pnsn_rainier_events["etype"].value_counts())


PNSN events detected by at least one Rainier station: 4355
etype
su    3311
px    1044
Name: count, dtype: int64


In [53]:
pnsn_rainier_events

,evid,orig_time_true,lat,lon,depth,etype,totalarr
0,10788008,2010-01-13 21:00:35.400000+00:00,47.167167,-121.817000,-0.867,px,7.0
1,10788193,2010-01-14 23:07:01.360000+00:00,46.961000,-122.074833,-1.285,px,7.0
2,10789093,2010-01-17 22:16:34.720000+00:00,47.586833,-120.754000,6.693,px,36.0
3,10790463,2010-01-22 20:38:54.870000+00:00,46.610833,-122.249500,-0.446,px,6.0
4,10792163,2010-01-28 23:53:01.810000+00:00,48.094667,-121.924500,1.052,px,8.0
...,...,...,...,...,...,...,...
4350,62244981,2026-04-21 00:53:38.410000+00:00,46.603000,-120.677167,-0.660,px,17.0
4351,62245161,2026-04-21 19:03:07.480000+00:00,48.472167,-122.638333,-0.080,px,22.0
4352,62245671,2026-04-22 15:20:23.662448+00:00,46.835600,-121.733000,-3.076,su,0.0
4353,62246211,2026-04-22 19:51:48.189999+00:00,46.834167,-121.732333,-2.270,su,5.0


In [55]:
import pandas as pd
import numpy as np

# load master catalog
master = pd.read_csv("../src/catalog_output/master_catalog.csv")
master["rounded_start"] = pd.to_datetime(master["rounded_start"], utc=True, format = 'mixed')

WINDOW_SEC = 60  # match within 60 seconds

print(f"QuakeXNet catalog: {len(master):,} events")
print(f"PNSN Rainier events: {len(pnsn_rainier_events):,}")

# ── match PNSN events to QuakeXNet catalog by time proximity ──────────────────
matches = []
for _, pnsn_row in pnsn_rainier_events.iterrows():
    t    = pnsn_row["orig_time_true"]
    diff = (master["rounded_start"] - t).abs()
    within = master[diff <= pd.Timedelta(seconds=WINDOW_SEC)]

    if len(within) > 0:
        best_idx = diff[within.index].idxmin()
        best     = master.loc[best_idx]
        matches.append({
            "evid":            pnsn_row["evid"],
            "pnsn_time":       t,
            "pnsn_etype":      pnsn_row["etype"],
            "pnsn_lat":        pnsn_row["lat"],
            "pnsn_lon":        pnsn_row["lon"],
            "event_id":        best["event_id"],
            "qxn_time":        best["rounded_start"],
            "qxn_class":       best["most_common_class"],
            "qxn_vote_margin": best["vote_margin"],
            "qxn_mean_max":    best["mean_max"],
            "time_diff_sec":   diff[best_idx].total_seconds(),
            "matched":         True,
        })
    else:
        matches.append({
            "evid":       pnsn_row["evid"],
            "pnsn_time":  t,
            "pnsn_etype": pnsn_row["etype"],
            "pnsn_lat":   pnsn_row["lat"],
            "pnsn_lon":   pnsn_row["lon"],
            "event_id":   None,
            "qxn_time":   None,
            "qxn_class":  None,
            "qxn_vote_margin": None,
            "qxn_mean_max":    None,
            "time_diff_sec":   None,
            "matched":    False,
        })

matches_df = pd.DataFrame(matches)
matched    = matches_df[matches_df["matched"]]
unmatched  = matches_df[~matches_df["matched"]]

print(f"\n=== MATCHING RESULTS ===")
print(f"PNSN events matched in QuakeXNet : {len(matched):,}  "
      f"({100*len(matched)/len(pnsn_rainier_events):.1f}%)")
print(f"PNSN events missed by QuakeXNet  : {len(unmatched):,}  "
      f"({100*len(unmatched)/len(pnsn_rainier_events):.1f}%)")

print(f"\n=== RECALL BY CLASS ===")
for etype in ["su", "px"]:
    total   = (pnsn_rainier_events["etype"] == etype).sum()
    matched_n = matched[matched["pnsn_etype"] == etype].shape[0]
    print(f"  {etype}: {matched_n}/{total} = {100*matched_n/total:.1f}% recall")

print(f"\n=== QuakeXNet CLASS vs PNSN CLASS (matched events only) ===")
confusion = pd.crosstab(
    matched["pnsn_etype"],
    matched["qxn_class"],
    margins=True
)
print(confusion)

print(f"\n=== NOVEL DETECTIONS (in QuakeXNet but not in PNSN) ===")
matched_qxn_ids = set(matched["event_id"].dropna())
novel = master[~master["event_id"].isin(matched_qxn_ids)]
print(f"Total novel detections: {len(novel):,}")
print(novel["most_common_class"].value_counts())

QuakeXNet catalog: 656,973 events
PNSN Rainier events: 4,355

=== MATCHING RESULTS ===
PNSN events matched in QuakeXNet : 2,842  (65.3%)
PNSN events missed by QuakeXNet  : 1,513  (34.7%)

=== RECALL BY CLASS ===
  su: 2033/3311 = 61.4% recall
  px: 809/1044 = 77.5% recall

=== QuakeXNet CLASS vs PNSN CLASS (matched events only) ===
qxn_class    eq   px    su   All
pnsn_etype                      
px           38  559   212   809
su           72  307  1654  2033
All         110  866  1866  2842

=== NOVEL DETECTIONS (in QuakeXNet but not in PNSN) ===
Total novel detections: 654,134
most_common_class
su    397513
eq    202124
px     54497
Name: count, dtype: int64


In [56]:
# 1. time difference distribution for matched events
print("Time difference distribution (seconds):")
print(matched["time_diff_sec"].describe())

# 2. what fraction of missed PNSN events are low-quality?
# (few arrivals = harder to detect)
unmatched_evids = set(unmatched["evid"])
pnsn_unmatched = pnsn_rainier_events[
    pnsn_rainier_events["evid"].isin(unmatched_evids)
]
print(f"\nMissed events — totalarr distribution:")
print(pnsn_unmatched["totalarr"].describe())
print(f"\nMatched events — totalarr distribution:")
matched_evids = set(matched["evid"])
pnsn_matched = pnsn_rainier_events[
    pnsn_rainier_events["evid"].isin(matched_evids)
]
print(pnsn_matched["totalarr"].describe())

# 3. recall by year — does it improve over time as network grows?
matched["year"] = pd.to_datetime(matched["pnsn_time"]).dt.year
pnsn_rainier_events["year"] = pd.to_datetime(
    pnsn_rainier_events["orig_time_true"]).dt.year

recall_by_year = []
for year in sorted(pnsn_rainier_events["year"].unique()):
    total     = (pnsn_rainier_events["year"] == year).sum()
    matched_n = (matched["year"] == year).sum()
    recall_by_year.append({
        "year": year,
        "total_pnsn": total,
        "matched": matched_n,
        "recall": matched_n / total if total > 0 else 0
    })

recall_df = pd.DataFrame(recall_by_year)
print("\nRecall by year:")
print(recall_df.to_string(index=False))

Time difference distribution (seconds):
count    2842.000000
mean       37.642149
std        18.379191
min         0.000000
25%        20.133506
50%        45.335062
75%        53.000655
max        59.981421
Name: time_diff_sec, dtype: float64

Missed events — totalarr distribution:
count    1513.000000
mean        3.194977
std         6.914801
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max        45.000000
Name: totalarr, dtype: float64

Matched events — totalarr distribution:
count    2841.000000
mean        5.643083
std         9.622123
min         0.000000
25%         0.000000
50%         0.000000
75%        10.000000
max        70.000000
Name: totalarr, dtype: float64

Recall by year:
 year  total_pnsn  matched   recall
 2010          58       27 0.465517
 2011         136      105 0.772059
 2012         152       89 0.585526
 2013         238      178 0.747899
 2014         207      124 0.599034
 2015         193      127 0.658031
 2016   

/tmp/ipykernel_665886/2177525224.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched["year"] = pd.to_datetime(matched["pnsn_time"]).dt.year


In [99]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.geodetics import locations2degrees, degrees2kilometers
import pandas as pd

# -------------------------------
# CONFIG
# -------------------------------
VOLCANO_NAME = "Mount Rainier"
VOLC_LAT = 46.8523
VOLC_LON = -121.7603
MAX_RADIUS_KM = 70

# Convert km to degrees (~111 km per degree)
MAX_RADIUS_DEG = MAX_RADIUS_KM / 111.0

# Time window (you can adjust)
STARTTIME = UTCDateTime("2000-01-01")
ENDTIME = UTCDateTime()

# -------------------------------
# FETCH STATIONS FROM IRIS
# -------------------------------
client = Client("IRIS")

inventory = client.get_stations(
    latitude=VOLC_LAT,
    longitude=VOLC_LON,
    maxradius=MAX_RADIUS_DEG,
    level="station",
    starttime=STARTTIME,
    endtime=ENDTIME
)

# -------------------------------
# PROCESS STATIONS
# -------------------------------
rows = []

for network in inventory:
    for station in network:
        st_lat = station.latitude
        st_lon = station.longitude
        st_ele = station.elevation

        # Compute distance in km
        dist_deg = locations2degrees(VOLC_LAT, VOLC_LON, st_lat, st_lon)
        dist_km = degrees2kilometers(dist_deg)

        if dist_km <= MAX_RADIUS_KM:
            rows.append({
                "Volcano_Name": VOLCANO_NAME,
                "Network": network.code,
                "Station": station.code,
                "Latitude": st_lat,
                "Longitude": st_lon,
                "Elevation": st_ele,
                "Distance_from_volc": dist_km,
                "Start": station.start_date,
                "End": station.end_date
            })

# -------------------------------
# SAVE TO CSV
# -------------------------------
df = pd.DataFrame(rows)

df.sort_values("Distance_from_volc", inplace=True)

output_file = "rainier_stations_within_70km.csv"
df.to_csv(output_file, index=False)

print(f"Saved {len(df)} stations to {output_file}")

Saved 1323 stations to rainier_stations_within_70km.csv
